## How to run this notebook

This notebook reads vector JSON shards produced by `fe-vector-embedding-v1.ipynb`, groups adjacent keyframes into scene/event records, and uploads event JSON shards back to GCS.

1. Run `fe-vector-embedding-v1.ipynb` first and copy the vector run `manifest_uri`.
2. Create Kaggle Secrets named `GCS_BUCKET` and `GCS_SERVICE_ACCOUNT_JSON`.
3. Run `Install dependencies`.
4. Edit the `Parameters` cell for `VECTOR_MANIFEST_URI` or `VECTOR_INPUT_PREFIX`, output prefix, filters, and event thresholds. Do not paste credentials into the notebook.
5. Run `Dry run` to confirm the notebook can read vector part files and sample embeddings.
6. Run `Demo one batch` to segment one video and inspect the uploaded event JSON.
7. When the demo output is correct, set `RUN_FULL = True` in the `Full run` cell and run the full job.
8. Outputs are written to `gs://<GCS_BUCKET>/<OUTPUT_PREFIX>/<run_id>/events/*.json` and `manifest.json`. Use these files to format rows for `events` and `event_keyframes`.


# FE Scene Detection v1

Note: This notebook runs after `fe-vector-embedding-v1`; it reads vector JSON shards from GCS and writes event JSON shards to GCS.


## Install dependencies

Note: Scene detection uses NumPy on CPU plus GCS I/O.


In [ ]:
!pip install -q google-cloud-storage tqdm pandas numpy


## Parameters

Note: Change only this cell for vector input prefix/manifest, output prefix, event thresholds, and filters.


In [ ]:
from pathlib import Path
import os

TASK_NAME = "scene_detection"
KAGGLE_SECRET_GCS_BUCKET = "GCS_BUCKET"
KAGGLE_SECRET_GCS_SERVICE_ACCOUNT_JSON = "GCS_SERVICE_ACCOUNT_JSON"


def read_kaggle_secret(secret_name: str, default: str = "") -> str:
    """Read a Kaggle Secret, with an environment variable fallback for local testing."""
    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(secret_name)
        if value:
            return value
    except Exception:
        pass
    return os.getenv(secret_name, default)


GCS_BUCKET = read_kaggle_secret(KAGGLE_SECRET_GCS_BUCKET)
GCS_SERVICE_ACCOUNT_JSON = read_kaggle_secret(KAGGLE_SECRET_GCS_SERVICE_ACCOUNT_JSON)
GCS_SERVICE_ACCOUNT_JSON_PATH = ""  # Optional local fallback path outside Kaggle.
VECTOR_INPUT_PREFIX = "processed/feature-annotations/fe-vector-embedding-v1/YOUR_VECTOR_RUN_ID"
VECTOR_MANIFEST_URI = ""
OUTPUT_PREFIX = "processed/feature-annotations/fe-scene-detection-v1"
GCS_TIMEOUT = 60

DATASET_CODE = "aic-2026"
DATASET_VERSION = "v1"
VIDEO_IDS = []
MAX_VECTOR_RECORDS = 0
MAX_VIDEOS = 0
DRY_RUN_SAMPLE = 5

MAX_TIME_GAP_SEC = 6.0
SCENE_SIMILARITY_THRESHOLD = 0.72
MAX_EVENT_DURATION_SEC = 45.0
SEGMENTATION_VERSION = "scene-openclip-threshold-v1"
EVENT_EMBEDDING_ROUND_DECIMALS = 6
OUTPUT_SHARD_SIZE = 256

LOCAL_OUTPUT_DIR = Path("/kaggle/working/fe-scene-detection-v1")


## Imports and runtime setup

Note: This cell loads common libraries, configures logging, and records the start timestamp.


In [ ]:
from __future__ import annotations

import csv
import json
import logging
import re
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path, PurePosixPath
from typing import Any, Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s - %(message)s")
LOGGER = logging.getLogger(TASK_NAME)


def utc_now() -> str:
    """Return the current UTC timestamp as an ISO-8601 string."""
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def make_run_id(task_name: str) -> str:
    """Create a unique run id for local and GCS output folders."""
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    return f"{task_name}-{stamp}-{uuid.uuid4().hex[:8]}"


print("Task:", TASK_NAME)
print("Started:", utc_now())


## GCS vector input and event helpers

Note: These helpers read vector JSON shards, segment one video at a time, and upload event JSON shards.


In [ ]:
def make_storage_client():
    """Create a GCS client from Kaggle Secret JSON, local JSON path, or default auth."""
    from google.cloud import storage
    from google.oauth2 import service_account

    if not GCS_BUCKET:
        raise ValueError("Missing Kaggle Secret `GCS_BUCKET` or environment variable GCS_BUCKET.")

    raw_json = (GCS_SERVICE_ACCOUNT_JSON or "").strip()
    if raw_json:
        if raw_json.startswith("{"):
            info = json.loads(raw_json)
            credentials = service_account.Credentials.from_service_account_info(info)
            return storage.Client(credentials=credentials, project=info.get("project_id"))
        if Path(raw_json).exists():
            return storage.Client.from_service_account_json(raw_json)

    if GCS_SERVICE_ACCOUNT_JSON_PATH and Path(GCS_SERVICE_ACCOUNT_JSON_PATH).exists():
        return storage.Client.from_service_account_json(GCS_SERVICE_ACCOUNT_JSON_PATH)

    return storage.Client()



def parse_gcs_uri(uri: str) -> tuple[str, str]:
    """Split a gs://bucket/path URI into bucket and blob path."""
    if not uri.startswith("gs://"):
        raise ValueError(f"Expected gs:// URI, got {uri}")
    bucket, blob = uri[len("gs://") :].split("/", 1)
    return bucket, blob


def download_text(client, gcs_uri: str) -> str:
    """Download text content from a GCS URI."""
    bucket, blob = parse_gcs_uri(gcs_uri)
    return client.bucket(bucket).blob(blob).download_as_text(timeout=GCS_TIMEOUT)


def upload_json_text(client, gcs_path: str, payload: str) -> str:
    """Upload JSON text to GCS and return the uploaded gs:// URI."""
    client.bucket(GCS_BUCKET).blob(gcs_path).upload_from_string(payload, content_type="application/json")
    return f"gs://{GCS_BUCKET}/{gcs_path}"


def list_vector_part_uris(client) -> list[str]:
    """List vector embedding JSON part files from manifest or prefix."""
    if VECTOR_MANIFEST_URI:
        manifest = json.loads(download_text(client, VECTOR_MANIFEST_URI))
        return [part["gcs_uri"] for part in manifest.get("output_parts", [])]
    prefix = VECTOR_INPUT_PREFIX.strip("/") + "/annotations/"
    return sorted(f"gs://{GCS_BUCKET}/{blob.name}" for blob in client.list_blobs(GCS_BUCKET, prefix=prefix, timeout=GCS_TIMEOUT) if blob.name.endswith(".json"))


def parse_records_payload(text: str) -> list[dict[str, Any]]:
    """Parse JSON array, JSON object with records, or JSONL text."""
    text = text.strip()
    if not text:
        return []
    if text.startswith("["):
        return json.loads(text)
    if text.startswith("{"):
        obj = json.loads(text)
        return obj.get("records", [obj]) if isinstance(obj, dict) else []
    return [json.loads(line) for line in text.splitlines() if line.strip()]


def iter_vector_records(client, part_uris: list[str]) -> Iterable[dict[str, Any]]:
    """Yield vector embedding records from GCS part files with optional filters."""
    selected_videos = set(VIDEO_IDS or []) or None
    seen = 0
    for uri in tqdm(part_uris, desc="Vector parts", unit="part"):
        for record in parse_records_payload(download_text(client, uri)):
            if record.get("task") != "vector_embedding":
                continue
            if selected_videos and record.get("video_id") not in selected_videos:
                continue
            yield record
            seen += 1
            if MAX_VECTOR_RECORDS and seen >= MAX_VECTOR_RECORDS:
                return


def extract_embedding(record: dict[str, Any]) -> np.ndarray:
    """Read an embedding vector from a vector annotation record."""
    vector = (record.get("annotation") or {}).get("embedding") or record.get("embedding")
    if vector is None:
        raise ValueError(f"Missing embedding for {record.get('keyframe_id')}")
    return np.asarray(vector, dtype=np.float32)


def normalized_video_rows(records: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], np.ndarray]:
    """Sort one video's records and return L2-normalized embedding matrix."""
    rows = sorted(records, key=lambda r: (float(r.get("frame_seconds") or 0.0), int(r.get("frame_idx") or 0), str(r.get("keyframe_id") or "")))
    vectors = np.stack([extract_embedding(row) for row in rows]).astype(np.float32)
    vectors = vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)
    return rows, vectors


def vector_to_json(vector: np.ndarray) -> list[float]:
    """Convert an event vector to a compact JSON float list."""
    if EVENT_EMBEDDING_ROUND_DECIMALS is not None:
        vector = np.round(vector, EVENT_EMBEDDING_ROUND_DECIMALS)
    return vector.astype(float).tolist()


def close_event(video_id: str, event_order: int, indices: list[int], rows: list[dict[str, Any]], vectors: np.ndarray) -> dict[str, Any]:
    """Create one backend-compatible event record from grouped keyframe indices."""
    idx = np.asarray(indices, dtype=int)
    event_vector = vectors[idx].mean(axis=0)
    event_vector = event_vector / max(float(np.linalg.norm(event_vector)), 1e-12)
    start = rows[indices[0]]
    end = rows[indices[-1]]
    middle = rows[indices[len(indices) // 2]]
    keyframe_ids = [rows[i].get("keyframe_id") for i in indices]
    event_id = f"{video_id}_E{event_order:04d}"
    return {
        "schema_version": "feature-extraction-v1",
        "task": TASK_NAME,
        "dataset": {"code": DATASET_CODE, "version": DATASET_VERSION},
        "event_id": event_id,
        "video_id": video_id,
        "event_order": event_order,
        "embedding_index_0": event_order,
        "start_seconds": float(start.get("frame_seconds") or 0.0),
        "end_seconds": float(end.get("frame_seconds") or 0.0),
        "start_frame": int(start.get("frame_idx") or 0),
        "end_frame": int(end.get("frame_idx") or 0),
        "start_frame_idx": int(start.get("frame_idx") or 0),
        "end_frame_idx": int(end.get("frame_idx") or 0),
        "representative_keyframe_id": middle.get("keyframe_id"),
        "representative_frame_id": middle.get("keyframe_id"),
        "n_keyframes": len(indices),
        "keyframe_ids": keyframe_ids,
        "keyframe_embedding_indices": indices,
        "keyframe_embedding_indices_raw": ",".join(str(i) for i in indices),
        "segmentation_version": SEGMENTATION_VERSION,
        "model_version": SEGMENTATION_VERSION,
        "event_embedding": vector_to_json(event_vector),
        "event_embedding_dim": int(event_vector.shape[0]),
        "json_value": {"max_time_gap_sec": MAX_TIME_GAP_SEC, "scene_similarity_threshold": SCENE_SIMILARITY_THRESHOLD, "max_event_duration_sec": MAX_EVENT_DURATION_SEC},
        "created_at": utc_now(),
    }


def segment_video(video_id: str, records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Group sorted keyframes into scene/event records using time and cosine thresholds."""
    if not records:
        return []
    rows, vectors = normalized_video_rows(records)
    events = []
    current = [0]
    current_vector = vectors[0].copy()
    for i in range(1, len(rows)):
        prev_time = float(rows[i - 1].get("frame_seconds") or 0.0)
        cur_time = float(rows[i].get("frame_seconds") or 0.0)
        start_time = float(rows[current[0]].get("frame_seconds") or 0.0)
        similarity = float(np.dot(current_vector, vectors[i]))
        should_split = (cur_time - prev_time > MAX_TIME_GAP_SEC) or (similarity < SCENE_SIMILARITY_THRESHOLD)
        if MAX_EVENT_DURATION_SEC is not None and cur_time - start_time > MAX_EVENT_DURATION_SEC:
            should_split = True
        if should_split:
            events.append(close_event(video_id, len(events), current, rows, vectors))
            current = [i]
            current_vector = vectors[i].copy()
        else:
            current.append(i)
            current_vector = vectors[np.asarray(current, dtype=int)].mean(axis=0)
            current_vector = current_vector / max(float(np.linalg.norm(current_vector)), 1e-12)
    events.append(close_event(video_id, len(events), current, rows, vectors))
    return events


def upload_event_part(client, records: list[dict[str, Any]], run_id: str, part_index: int) -> dict[str, Any]:
    """Save one event JSON shard locally and upload it to GCS."""
    out_dir = LOCAL_OUTPUT_DIR / run_id / "events"
    out_dir.mkdir(parents=True, exist_ok=True)
    filename = f"part-{part_index:06d}.json"
    payload = json.dumps(records, ensure_ascii=False)
    local_path = out_dir / filename
    local_path.write_text(payload, encoding="utf-8")
    started = time.perf_counter()
    gcs_uri = upload_json_text(client, f"{OUTPUT_PREFIX.strip('/')}/{run_id}/events/{filename}", payload)
    return {"part_index": part_index, "records": len(records), "local_path": str(local_path), "gcs_uri": gcs_uri, "upload_seconds": round(time.perf_counter() - started, 3)}


def write_manifest(client, run_id: str, manifest: dict[str, Any]) -> str:
    """Write and upload the scene detection manifest."""
    out_dir = LOCAL_OUTPUT_DIR / run_id
    out_dir.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(manifest, ensure_ascii=False, indent=2)
    (out_dir / "manifest.json").write_text(payload, encoding="utf-8")
    return upload_json_text(client, f"{OUTPUT_PREFIX.strip('/')}/{run_id}/manifest.json", payload)


## Run helpers

Note: These helpers implement dry run, demo one video, and full scene detection with timing metrics.


In [ ]:
def run_dry_run() -> dict[str, Any]:
    """List vector parts and print a small sample of vector metadata."""
    client = make_storage_client()
    part_uris = list_vector_part_uris(client)
    sample = []
    for record in iter_vector_records(client, part_uris[:1]):
        sample.append({"video_id": record.get("video_id"), "keyframe_id": record.get("keyframe_id"), "frame_idx": record.get("frame_idx"), "embedding_dim": len((record.get("annotation") or {}).get("embedding") or [])})
        if len(sample) >= DRY_RUN_SAMPLE:
            break
    report = {"task": TASK_NAME, "mode": "dry_run", "parts": len(part_uris), "sample": sample, "created_at": utc_now()}
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return report


def flush_video(video_id: str, records: list[dict[str, Any]], pending: list[dict[str, Any]], video_metrics: list[dict[str, Any]]) -> None:
    """Segment one video and append event records plus timing metrics."""
    started = time.perf_counter()
    events = segment_video(video_id, records)
    seconds = time.perf_counter() - started
    pending.extend(events)
    video_metrics.append({"video_id": video_id, "keyframes": len(records), "events": len(events), "seconds": round(seconds, 3), "keyframes_per_second": round(len(records) / max(seconds, 1e-9), 3)})
    print(f"[{TASK_NAME}] video={video_id} keyframes={len(records)} events={len(events)} seconds={seconds:.2f}")


def run_scene_detection(mode: str, max_videos: int | None = None) -> dict[str, Any]:
    """Run scene segmentation from vector JSON shards and upload event JSON shards."""
    client = make_storage_client()
    part_uris = list_vector_part_uris(client)
    if not part_uris:
        raise RuntimeError("No vector JSON part files found. Check VECTOR_MANIFEST_URI or VECTOR_INPUT_PREFIX.")
    if mode == "demo":
        max_videos = 1
    run_id = make_run_id(TASK_NAME if mode == "full" else f"{TASK_NAME}-{mode}")
    started = time.perf_counter()
    current_video = None
    current_records: list[dict[str, Any]] = []
    completed_videos = 0
    pending: list[dict[str, Any]] = []
    parts: list[dict[str, Any]] = []
    video_metrics: list[dict[str, Any]] = []
    part_index = 1

    for record in iter_vector_records(client, part_uris):
        video_id = str(record.get("video_id"))
        if current_video is None:
            current_video = video_id
        if video_id != current_video:
            flush_video(current_video, current_records, pending, video_metrics)
            completed_videos += 1
            if len(pending) >= OUTPUT_SHARD_SIZE:
                uploaded = upload_event_part(client, pending, run_id, part_index)
                parts.append(uploaded)
                print(f"[{TASK_NAME}] uploaded part={part_index} events={uploaded['records']} uri={uploaded['gcs_uri']}")
                pending = []
                part_index += 1
            if (max_videos is not None and completed_videos >= max_videos) or (MAX_VIDEOS and completed_videos >= MAX_VIDEOS):
                break
            current_video = video_id
            current_records = []
        current_records.append(record)

    if current_video is not None and current_records and not ((max_videos is not None and completed_videos >= max_videos) or (MAX_VIDEOS and completed_videos >= MAX_VIDEOS)):
        flush_video(current_video, current_records, pending, video_metrics)
        completed_videos += 1
    if pending:
        uploaded = upload_event_part(client, pending, run_id, part_index)
        parts.append(uploaded)
        print(f"[{TASK_NAME}] uploaded part={part_index} events={uploaded['records']} uri={uploaded['gcs_uri']}")

    elapsed = time.perf_counter() - started
    total_keyframes = sum(row["keyframes"] for row in video_metrics)
    total_events = sum(row["events"] for row in video_metrics)
    manifest = {
        "schema_version": "feature-extraction-v1",
        "task": TASK_NAME,
        "mode": mode,
        "run_id": run_id,
        "created_at": utc_now(),
        "dataset": {"code": DATASET_CODE, "version": DATASET_VERSION},
        "input": {"bucket": GCS_BUCKET, "vector_input_prefix": VECTOR_INPUT_PREFIX, "vector_manifest_uri": VECTOR_MANIFEST_URI, "video_ids": VIDEO_IDS},
        "parameters": {"max_time_gap_sec": MAX_TIME_GAP_SEC, "scene_similarity_threshold": SCENE_SIMILARITY_THRESHOLD, "max_event_duration_sec": MAX_EVENT_DURATION_SEC, "segmentation_version": SEGMENTATION_VERSION, "output_shard_size": OUTPUT_SHARD_SIZE},
        "totals": {"videos": completed_videos, "keyframes": total_keyframes, "events": total_events, "parts": len(parts)},
        "timing": {"elapsed_seconds": round(elapsed, 3), "keyframes_per_second": round(total_keyframes / max(elapsed, 1e-9), 3)},
        "output_parts": parts,
        "video_metrics": video_metrics,
        "supabase_schema_hint": {"event_table": "events", "event_keyframe_table": "event_keyframes", "join_key": "event_id/keyframe_id"},
    }
    manifest_uri = write_manifest(client, run_id, manifest)
    manifest["manifest_uri"] = manifest_uri
    print(json.dumps({"run_id": run_id, "manifest_uri": manifest_uri, "totals": manifest["totals"], "timing": manifest["timing"]}, ensure_ascii=False, indent=2))
    return manifest


def run_demo_batch() -> dict[str, Any]:
    """Run scene detection for one video."""
    return run_scene_detection(mode="demo", max_videos=1)


def run_full() -> dict[str, Any]:
    """Run scene detection for all configured videos."""
    return run_scene_detection(mode="full", max_videos=None)


## Dry run

Note: This cell only lists GCS frame metadata. It does not download frames, load models, or upload output.


In [ ]:
dry_report = run_dry_run()


## Demo one batch

Note: This cell processes one small batch, uploads one JSON shard, and writes a manifest. Run this before the full job.


In [ ]:
demo_report = run_demo_batch()


## Full run

Note: Set `RUN_FULL = True` only after dry run and demo are OK. This may run for hours.


In [ ]:
RUN_FULL = False

if RUN_FULL:
    full_report = run_full()
else:
    print("Set RUN_FULL = True in this cell to process all configured frames.")


## Output check

Note: This cell lists local manifest files created in the Kaggle working directory.


In [ ]:
for path in sorted(LOCAL_OUTPUT_DIR.rglob("manifest.json")):
    print(path)
